##今週やるべきものは、データをSQLiteのDBに移すこと
    ##W2まで： crawler → pages_w2.json（ファイル） → search（JSONを読む）
    ##W3から： crawler → DB（SQLite）          → search（DBを読む）

##具体的には、
  ##1.器を変える　器の設計（schema)→読み書き（database)
  ##2.本文まで探す（全文検索）　→DBに本文が入っているので、本文まで検索できる。
  ##3.検索の読み込み先をDBに移し替えて、完成。

##今週のStepは以下4つ
  ##Step:1 器の設計（Scheema.sql)
  ##Step2:DBに読み書き
  ##Step3:全体検索を作成
  ##Step4:検索の読み込み先をDBに差し替え

##内容：
  ##① スクショ1枚：検索結果が出ている画面
  ##② pages.json のままでも、アプリは動きます。それでも DB に移したのはなぜですか？ 自分の言葉で説明してください（3〜5行）
  ##ーpages.json のままだと何が困るのか。
  ##ーDBにすると何が解決するのか。
  ##ーなぜ今回はSQLite？他にどんなDBMSがあり、それぞれどういう使い分けをする？　など
  ##③ 今週いちばん詰まったところ（2〜3行）
  ##どこで止まって、どう抜けたか。抜けられていなければ「抜けていない」でOK。

Step1:器を設計する
以下のコードを記載することで、フォルダ上にSQLファイルを書き出す。

In [2]:
schema_sql = """-- schema.sql
-- Tech0 Search データベース設計（W#:データ基盤の最小構成）

--pagesテーブル（メイン）
--UNIQUE 制約：同じURは重複登録が不可
CREATE  TABLE IF NOT EXISTS pages (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    url         TEXT NOT NULL UNIQUE,
    title       TEXT NOT NULL,
    description TEXT,
    full_text   TEXT,
    author      TEXT,
    category    TEXT,
    keywords    TEXT,
    word_count  INTEGER DEFAULT 0,
    crawled_at  DATETIME,
    created_at  DATETIME DEFAULT CURRENT_TIMESTAMP,
    updated_at  DATETIME DEFAULT CURRENT_TIMESTAMP
);

--keywordsテーブル（TF-IDFスコア保存用。実際に使うのは W4）
CREATE TABLE IF NOT EXISTS keywords (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    page_id     INTEGER NOT NULL,
    keyword     TEXT NOT NULL,
    tf_score    REAL DEFAULT 0.0,
    tfidf_score REAL DEFAULT 0.0,
    FOREIGN KEY (page_id) REFERENCES pages(id) ON DELETE CASCADE

);

-- インデックス作成（検索を高速化）
CREATE INDEX IF NOT EXISTS idx_keyword ON keywords(keyword);
CREATE INDEX IF NOT EXISTS idx_page_id ON keywords(page_id);

-- ※ search_logs / click_logs テーブルは W6（発展課題）で追加します。
"""

# schema/sqlファイルに書き出す
with open("schema.sql" , "w" , encoding="utf-8") as f:
    f.write(schema_sql)

print("schema/sqlを作成しました（pages / keywords) ")

schema/sqlを作成しました（pages / keywords) 


Step2 
2_1.database.pyを作る
scheema.sqlというSQLのファイルができても、Pythonから読み書きできないといけない。
DBとの窓口をdatabase.pyの1ファイルに集約して、アプリ側がSQLを書かなくても済む形にする

#ここで確認すべきこと
1.get_connection()でDBへ接続している
2.init_db()でschema.sqlを読み込んでDBを初期化している
3.data/tech0_search.dbファイルが生成されていることを確認する。

In [32]:
import sqlite3
from pathlib import Path
from datetime import datetime

# DBファイルのパス（data/サブフォルダに保存する）
DB_PATH = Path("data/tech0_search.db")

def get_connection():
    """
     DBへの接続を取得する。

     row factoryを設定することで、行データを辞書のように扱える。
     data/ フォルダが存在しない場合には自動で作成する。
    """
    DB_PATH.parent.mkdir(exist_ok=True) #data/フォルダがなければ生成
    conn = sqlite3.connect(str(DB_PATH))
    conn.row_factory = sqlite3.Row      #データを辞書のように扱う
    return conn

def init_db():
    """
    scheema.sqlを読み込んでDBを初期化する。

    CREATE TSBLE IF NOT EXISTSを使っているので、
    すでにテーブルが存在する場合は何もしない。

    """
    conn = get_connection()
    with open("schema.sql", "r", encoding="utf-8") as f:
        conn.executescript(f.read())    #SQLファイルをまとめて実行
    conn.commit()
    conn.close()
    print("DB を初期化しました（data/tech0_serach.db)")

# 動作確認
init_db()

DB を初期化しました（data/tech0_serach.db)


Step2
2_2. INSERT OR REPLACEでページを登録する
┗INSERT OR REPLACEは、同じURLのデータがあれば上書き、なければ新規追加するという意味

##確認すべきこと
1.INSERT OR REPLACEは、重複チェックふようで簡潔にかける
2.insert_page(page: dict) -> でページ情報をDBに登録してる
3.get_all_pages() -> で全ページを、登録日時の新しい順で取得する。

INSERT OR REPLACE` は「削除してから再挿入」するため、実行のたびに id が増加します。
`data/tech0_search.db` はノートブックと同じフォルダに**ファイルとして残り続ける**ため、
リセットしたい場合は次の方法を使ってください：
 ```python
 import os
 os.remove("data/tech0_search.db")  # DB ファイルを削除して最初からやり直す

In [33]:
def _keywords_to_text(keywords) -> str:
    """ keywords(リスト or 文字列)を DB 保存用のカンマ区切り文字列にすする"""
    if not keywords:
        return ""
    if isinstance(keywords, str):
        return keywords
    return ",".join(keywords)

def insert_page(page: dict) -> int:
    """
    ページ情報を DB に登録する。

    INSERT OR REPLACE:同じURLのデータがあれば上書き、なければ新規追加。
    これにより、「同じページを再クロールした時に最新データに更新できる」

    Args:
        page: ページ情報の辞書(crawl_url()の返り値と同形式)
    
    Return:
        登録された行の id

    """
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        INSERT OR REPLACE INTO pages
            (url, title, description, full_text, author, category, keywords, word_count, crawled_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        page["url"],
        page["title"],
        page.get("description",""),
        page.get("full_text",""),
        page.get("author",""),
        page.get("category",""),
        _keywords_to_text(page.get("keywords")), #リスト → "DX,営業店"にして保存
        page.get("word_count",0),
        page.get("crawled_at",datetime.now().isoformat()),
    ))

    page_id = cursor.lastrowid #登録された行のIDを取得する
    conn.commit()
    conn.close()
    return page_id

def get_all_pages() -> list:
    """全ページを登録日時の新しい順で取得する。"""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM pages ORDER BY created_at DESC")
    rows = cursor.fetchall()
    conn.close()
    pages = []
    for row in rows:
        pg =dict(row)
        # 保存時にカンマ区切りにしたkeywordsを、リストに戻す
        pg["keywords"] = [k.strip() for k in (pg.get("keywords") or "").split(",") if k.strip()]
        pages.append(pg)
    return pages #sqlite3.rowを辞書に変換して返す

#動作確認：テストデータを登録する
test_pages = [
    {"url":"https://example.com/dx-report",
     "title":"DX戦略レポート2024",
     "description":"営業店のDX戦略について解説した報告書",
     "full_text":"DX 推進 で 営業店 の 効率 が 向上 デジタル 変革 事例",
     "keywords":"DX,営業店,デジタル変革",
     "word_count":150,
     "crawled_at":"2024-01-15T10:00:00"},
    {"url":"https://example.com/ai-guide",
     "title":"AI活用ガイド",
     "description":"業務効率化のためのAI導入ガイド",
     "full_text":"AI 活用 で 業務 効率化 を 進める 機械 学習 ツール",
     "keywords":"AI,機械学習,業務効率化",
     "word_count":120,
     "crawled_at":"2024-02-20T15:00:00"},
]

for p in test_pages:
    pid = insert_page(p)
    print(f"登録完了：id={pid} {p['title']}")

#登録されたデータを確認する
all_pages = get_all_pages()
print(f"\n現在の登録ページ数：{len(all_pages)}件")


def log_search(query: str, results_count: int,user_id: str = None) -> int:
    """
    検索ログを記録する（W6 の発展課題で実装予定）
    現在はスタブ（空の関数）として定義しています。
    W6 の発展課題で実際の記録処理を実装します。
    """
    pass # W6の発展課題にて実装

登録完了：id=1 DX戦略レポート2024
登録完了：id=2 AI活用ガイド

現在の登録ページ数：2件


In [34]:
import json #jsonファイルを読むために使う（W2と同じ）

def migrate_from_json(json_path: str = "pages_w2.json") -> int:
    """
    W2で使っていたJSONのデータを、丸ごとDBに移す

    Args:
        json_path: 移行元のJSONファイル

    Returns:
        移行した件数
    """

    # ① JSONファイルを読み込む（W2でやったものと同じ）
    with open(json_path, "r", encoding="utf-8") as f:
        pages = json.load(f)

    # ② 1件ずつDBに入れる（　Step 2-2で作った　insert_pageを使うだけ
    for p in pages:
        insert_page(p)

    return len(pages)

# --実行：W2のデータをDBに移す--
n = migrate_from_json("pages_w2.json")
print(f"✅ {n}件を DB に移行しました")

# --移行後の DB の中身を確認する --
pages = get_all_pages()
print(f"現在の DB の登録件数：{len(pages)} 件（テストデータ2件 ＋ 移行分）")
print()
for p in pages[:5]:
    honbun = "本文あり" if p.get("full_text") else "本文なし"
    print(f" id={p['id']:>2} [{honbun}] {p['title'][:30]}")

✅ 25件を DB に移行しました
現在の DB の登録件数：26 件（テストデータ2件 ＋ 移行分）

 id= 3 [本文なし] 営業店DX推進レポート
 id= 4 [本文なし] 住宅ローン審査AI導入の検討
 id= 5 [本文なし] 法人融資 稟議書の書き方ガイド
 id= 6 [本文なし] AIチャットボットによる照会対応DX
 id= 7 [本文なし] 全店DX推進ロードマップ2026


##ここで確認すべきこと
1.現在DBには、本文あり、本文なしが混在する
2.もう一度同じ処理をしても、データ自体は重複しない。
　本当にそうなるのかを、以下のコードで試してみる。

In [35]:
#移行前の件数を記録
before = len(get_all_pages())

#全く同じJsonをもう一度試してみる。
migrate_from_json("pages_w2.json")

after = len(get_all_pages())

print(f"1回目の移行後：{before} 件")
print(f"2回目の移行後：{after} 件")
print()

if before == after:
    print("件数が増えていない = 同じURLは新規登録ではなく、上書きされた")
    print(" W2(JSONにapppend)なら、同じデータが2セットに増えた")
else:
    print(f"{after-before}件増えてしまった(UNIQUE制約を確認)")

1回目の移行後：26 件
2回目の移行後：26 件

件数が増えていない = 同じURLは新規登録ではなく、上書きされた
 W2(JSONにapppend)なら、同じデータが2セットに増えた


In [36]:
# 移行前の件数を記録しておく
before = len(get_all_pages())

# まったく同じJSONを、もう一度移行してみる
migrate_from_json("pages_w2.json")

after = len(get_all_pages())

print(f"1回目の移行後: {before} 件")
print(f"2回目の移行後: {after} 件")
print()

if before == after:
    print("✅ 件数が増えていない ＝ 同じURLは新規追加ではなく【上書き】された")
    print("   W2（JSONにappend）なら、同じデータが2セットに増えていました。")
else:
    print(f"❌ {after - before} 件増えてしまった（UNIQUE制約を確認しよう）")

1回目の移行後: 26 件
2回目の移行後: 26 件

✅ 件数が増えていない ＝ 同じURLは新規追加ではなく【上書き】された
   W2（JSONにappend）なら、同じデータが2セットに増えていました。


Step3.全文検索をつくる
title + description + full_text + keywords` を1本の検索対象テキストにまとめられる

0. DBに移したデータを読み込む（`get_all_pages()`）
1. 検索対象の文字列を作る  
2. キーワードが何回出てきたかを数える（`match_count`）  
3. 検索結果を並べ替える  
4. preview（抜粋）を作る  
5. `search_fulltext.py` の形にまとめる

In [8]:
#3-0.DBに移したデータを読み込む
pages = get_all_pages()
print(f"DBから読み込んだページ数：{len(pages)} 件")
if pages:
    print("先頭ページ:", pages[0]["title"])

DBから読み込んだページ数：26 件
先頭ページ: 営業店DX推進レポート


In [9]:
#3-1.検索対象の文字列を作る
 ##検索をするには、まず「どこを対象にするか」を決める必要がある。
 ##今回は、1ページ分のデータから、title,description,full_text,keywordsを繋げて1つの検索文字列を作る

# まずは1ページ分のデータを使って、検索対象文字列を作ってみる
page = pages[0]

# title, description, full_text, keywords をまとめて1つの文字列にする
search_text = " ".join([
    page.get("title", ""),
    page.get("description", ""),
    page.get("full_text", ""),
    " ".join(page.get("keywords", [])),
])

print(search_text[:500])

営業店DX推進レポート 窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録  DX 営業店 業務効率


In [ ]:
#3-2-a.キーワードが含まれいるか調べてみる
# 検索キーワード
query = "DX"

# 大文字小文字の違いを吸収するために lower() にして比較する
found = query.lower() in search_text.lower()

print("検索キーワード:", query)
print("含まれているか:", found)

検索キーワード: DX
含まれているか: True


In [11]:
#3-2-b.何回でてくるか数えてみる 
  #今回はシンプルに、count()を使ってキーワードの出現回数を数える
  #これは後々、match_countとして扱う。
  #match_countが多いページほど、より関連が高いとみなし、上に表示するアルゴリズムに使う

count = search_text.lower().count(query.lower())

print("検索キーワード:", count)
print("出現回数:", count)

検索キーワード: 3
出現回数: 3


In [16]:
#3-2-c.全ページに対して検索してみる
#実際には、保存されている全ページ（全レコード）を順番に確認し、各ページ毎に以下の処理を行う
 #1.検索対象文字列を作成
 #2.キーワードの出現回数を数える
 #3.ヒットしたページだけ結果に追加

# 全てのページを1件ずつ確認
query = "DX"
results = []

for page in pages:

    # そのページの検索対象文字列を作る
    text = " ".join([
        page.get("title",""),
        page.get("description",""),
        page.get("full_text",""),
        " ".join(page.get("keywords",[])),
    ]).lower()

    # キーワードの出現回数を数える
    count = text.count(query.lower())

    #1回以上出てきたページだけ結果に追加
    if count > 0:
        result = page.copy()
        result["match_count"] = count
        results.append(result)

print("検索結果件数", len(results))
results[:2]

検索結果件数 7


[{'id': 411,
  'url': 'https://kb.techzeron-bank.example/dx-report',
  'title': '営業店DX推進レポート',
  'description': '窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録',
  'full_text': '',
  'author': '田中太郎',
  'category': '事例',
  'keywords': ['DX', '営業店', '業務効率'],
  'word_count': 0,
  'crawled_at': '2026-09-24T23:28:34.465550',
  'created_at': '2026-09-24 14:28:34',
  'updated_at': '2026-09-24 14:28:34',
  'match_count': 3},
 {'id': 412,
  'url': 'https://kb.techzeron-bank.example/ai-screening',
  'title': '住宅ローン審査AI導入の検討',
  'description': 'AI事前審査で融資業務をDX化する企画提案',
  'full_text': '',
  'author': '鈴木花子',
  'category': '企画',
  'keywords': ['AI', '融資', '審査'],
  'word_count': 0,
  'crawled_at': '2026-09-24T23:28:34.466862',
  'created_at': '2026-09-24 14:28:34',
  'updated_at': '2026-09-24 14:28:34',
  'match_count': 1}]

In [17]:
#3-3.検索結果を並び替える
#mtch_countの大きい順に並び替える
sorted_results = sorted(
    results,
    key=lambda x:x["match_count"],
    reverse=True

)

print("並び替え後の上位3件:")
for r in sorted_results[:3]:
    print(r["title"], "->", r["match_count"])


並び替え後の上位3件:
DX戦略レポート2024 -> 4
営業店DX推進レポート -> 3
AIチャットボットによる照会対応DX -> 3


In [ ]:
#3-4-a.previewを作る
 ##検索結果に、タイトルだけでなく本文の一部が表示されることが多い、これをPreviewと呼ぶ
 ##キーワード箇所の前後を切り出したプレビューを作る
def make_preview(text: str, query: str , ctx: int = 80) -> str:

    """
    キーワードが見つかった位置の前後を切り出し、previewを作る

    Args:
        text:本文
        query:検索キーワード
        ctx:前後に何文字表示するのか

    Returns:
        preview文字列

    """

    #キーワードの位置を探す
    pos = text.lower().find(query.lower())

    #見つからない場合は先頭200文字を返す
    if pos == -1:
        return text[:200]

    #前後の範囲を決める
    start = max(0, pos - ctx)
    end = min(len(text), pos + len(query) + ctx)

    #前後を切り出して返す
    return"..." + text[start:end] + "..."

In [ ]:
#3-4-b上記で作成したmake_previewを使い、検索キーワード周辺の本文を表示する
query = "DX"
results = []

for page in pages:

    #検索対象文字列を作る
    text = " ".join([
        page.get("title",""),
        page.get("description",""),
        page.get("full_text",""),
        "".join(page.get("keywords", [])),
    ]).lower()

    #出現回数を数える
    count = text.count(query.lower())

    #ヒットしたページだけ結果に追加
    if count > 0:
        result = page.copy()
        result["match_count"] = count

        #previewはfull_textがあればそれを使用。なけれbdescriptionを使用
        result["preview"] = make_preview(
            page.get("full_text") or page.get("description",""),
            query
        )

        results.append(result)

#match_count順に並べる
results = sorted(results, key=lambda x: x["match_count"], reverse=True)

print("検索結果件数:",len(results))
results[:2]

検索結果件数: 7


[{'id': 359,
  'url': 'https://example.com/dx-report',
  'title': 'DX戦略レポート2024',
  'description': '営業店のDX戦略について解説した報告書',
  'full_text': 'DX 推進 で 営業店 の 効率 が 向上 デジタル 変革 事例',
  'author': '',
  'category': '',
  'keywords': ['DX', '営業店', 'デジタル変革'],
  'word_count': 150,
  'crawled_at': '2024-01-15T10:00:00',
  'created_at': '2026-09-24 14:28:22',
  'updated_at': '2026-09-24 14:28:22',
  'match_count': 4,
  'preview': '...DX 推進 で 営業店 の 効率 が 向上 デジタル 変革 事例...'},
 {'id': 411,
  'url': 'https://kb.techzeron-bank.example/dx-report',
  'title': '営業店DX推進レポート',
  'description': '窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録',
  'full_text': '',
  'author': '田中太郎',
  'category': '事例',
  'keywords': ['DX', '営業店', '業務効率'],
  'word_count': 0,
  'crawled_at': '2026-09-24T23:28:34.465550',
  'created_at': '2026-09-24 14:28:34',
  'updated_at': '2026-09-24 14:28:34',
  'match_count': 3,
  'preview': '...窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録...'}]

3-4_c.検索ロジックを関数にまとめる
上記までで、全文検索のロジックはすでに完成している。
やっていることは
```
検索対象文字列を作る
↓
キーワード出現回数を数える
↓
ヒットしたページを results に追加
↓
preview を付ける
↓
match_count順に並べる
```

ただし、このままだと毎回同じコードを書く。
この処理が面倒なので、'serch_fulltext()'という関数にまとめる作業。

In [29]:
def make_preview(text: str, query: str, ctx:int = 80) -> str:
    """
    マッチ箇所終焉のプレビューを生成

    Args:
            text:本文
            query:検索キーワード
            ctx:前後に何文字表示するのか
    
        Returns:
            preview文字列
    """

    #textまたはqueryが空なら、空文字を返す
    if not text or not query:
        return ""

    #キーワードの位置を探す
    pos = text.lower().find(query.lower())

    #見つからない場合は前頭200文字を返す
    if pos == -1:
        return(text[:200] + "...") if len(text) > 200 else text

    #前後の切り出し範囲
    start = max(0, pos - ctx)
    end = min(len(text), pos + len(query) + ctx)

    preview = ""

    if start > 0:
        preview += "..."

    preview += text[start:end]

    if end < len(text):
        preview += "..."

    return preview

def search_fulltext(query: str,pages: list) -> list:
    """
    全文検索（本文含む）を実行し、マッチ数でスコアリングする。

    Args:
        query: 検索キーワード
        pages: ページリスト

    Returns:
        マッチしたページのリスト（スコア降順）
    """

    #空検索なら空リストを返す
    if not query.strip():
        return[]

    results = []

    #検索キーワードを小文字化
    q = query.lower()

    #全ページを順番に確認
    for page in pages:

        #検索対象テキストを作る
        text = " ".join([
            page.get("title",""),
            page.get("description",""),
            page.get("full_text",""),
            "".join(page.get("keywords", [])),
        ]).lower()

        #出現回数を数える
        count = text.count(q)

        #ヒットしたページだけ追加
        if count > 0:

            r = page.copy()

            #match_countを追加
            r["match_count"] = count

            #previewを追加
            r["preview"] = make_preview(
                page.get("full_text") or page.get("description", ""),
                query
            )

            results.append(r)

    #match_count順に並べる
    results.sort(key=lambda x: x["match_count"], reverse=True)

    return results

In [37]:
#3-5.serch fulltextを試してみる
 ##ここまでで、search_fulltextが完成しました。
 ##この関数を使い、検索が正しく動くかを確認する

 ##1.検索キーワードを指定
 ##2.search_fulltext()を実行
 ##3.検索結果の件数を確認
 ##4.上位の検索結果を表示

#検索キーワード
query = "DX"

#search_fulltextを実行
results = search_fulltext(query,pages)

#検索結果件数を表示
print("検索結果関数:",len(results))

#上位5件を表示
for r in results[:5]:
    print("----")
    #タイトル
    print("タイトル",r["title"])
    #キーワード出現回数
    print("match_count:", r["match_count"])
    #preview(本文の一部)
    print("preview:", r["preview"][:120])

検索結果関数: 7
----
タイトル DX戦略レポート2024
match_count: 4
preview: DX 推進 で 営業店 の 効率 が 向上 デジタル 変革 事例
----
タイトル 営業店DX推進レポート
match_count: 3
preview: 窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録
----
タイトル AIチャットボットによる照会対応DX
match_count: 3
preview: コールセンターにAIチャットボットを導入し照会対応を効率化したDX事例
----
タイトル 全店DX推進ロードマップ2026
match_count: 3
preview: 勘定系刷新とデジタルチャネル強化を柱とする全店DX計画の概要
----
タイトル 営業店ペーパーレス化 全店展開レポート（2026年度上期）
match_count: 2
preview: ... パーセント 削減 した 紙 の 稟議 書 を 電子 化 する こと で 決裁 までの 日数 も 短縮 され た 今後 は 融資 部門 への 展開 を 進める DX 推進 室 は 効果 測定 を 継続 する
